# 🧠 Cogito — Train Your Own Reasoning Model with GRPO

This notebook trains a small language model (**Qwen2.5-1.5B-Instruct**) to *reason* — to think step by step before answering — using **GRPO**, the reinforcement-learning method behind **DeepSeek-R1**.

**The task:** the *Countdown* numbers game — combine given numbers with `+ − × ÷` to hit a target (e.g. use 3, 7, 9, 50 to make 471).

**The "aha moment" we're chasing:**
- *Before* training, the base model guesses and rambles.
- *After* GRPO, the **same** model lays out a step-by-step plan and lands on the target far more often.

We build in **phases**. This is **Phase 0: Setup**.

---

### What is GRPO, in one paragraph?
GRPO (*Group Relative Policy Optimization*) is reinforcement learning for LLMs. For each problem the model generates a **group** of answers (say 8 attempts). A **reward function** scores each attempt (*did it hit the target? did it use the right format?*). GRPO then pushes the model **toward the attempts that scored above the group's average**, and away from the below-average ones. No labelled "correct reasoning" is needed — the model **discovers** that reasoning earns reward and starts doing it on its own. That discovery is the "aha".

## Phase 0 — Setup & GPU check

**Goal:** confirm we have a free **T4 GPU**, install **Unsloth + vLLM**, load **Qwen2.5-1.5B**, and run one sanity generation. Nothing is trained yet — we're just proving the machine works.

> ⚙️ **First, turn on the GPU:** menu **Runtime → Change runtime type → T4 GPU → Save**.

### Step 1 · Install Unsloth + vLLM

This cell looks fussy on purpose. GRPO generates many answers per problem, so it needs a **fast inference engine (vLLM)**. On a free **T4** (an older "Turing" GPU) only specific versions of vLLM/Triton still work, so the cell **detects the T4 and pins them** (`vllm==0.9.2`, `triton==3.2.0`). `%%capture` hides the long install log.

*Takes ~3–5 minutes.* If a later cell errors on an import, re-run this cell **without** the `%%capture` line to read the real error.

In [ ]:
%%capture
import os
os.environ["UNSLOTH_VLLM_STANDBY"] = "1"  # frees vLLM memory during training -> ~30% more usable context
!pip install --upgrade -qqq uv
if "COLAB_" not in "".join(os.environ.keys()):
    # Not on Colab/Kaggle: simple install
    !pip install unsloth vllm
else:
    try:
        import numpy, PIL
        _numpy = f"numpy=={numpy.__version__}"; _pil = f"pillow=={PIL.__version__}"
    except:
        _numpy = "numpy"; _pil = "pillow"
    try:
        import subprocess
        is_t4 = "Tesla T4" in str(subprocess.check_output(["nvidia-smi"]))
    except:
        is_t4 = False
    # T4 (Turing) -> pinned vllm 0.9.2 + triton 3.2.0; newer GPUs can take the latest
    _vllm, _triton = ("vllm==0.9.2", "triton==3.2.0") if is_t4 else ("vllm==0.15.1", "triton")
    !uv pip install -qqq --upgrade {_vllm} {_numpy} {_pil} torchvision bitsandbytes xformers unsloth
    !uv pip install -qqq {_triton}
    !uv pip install -qqq --no-deps --upgrade "torchao>=0.16.0"
!uv pip install -qqq transformers==4.56.2
!uv pip install -qqq --no-deps trl==0.22.2

### Step 2 · Confirm the GPU

You should see **Tesla T4**, ~**14.7 GB** of VRAM, and `bf16 OK? : False`. That `False` is expected — the T4 can't do bf16 math, so we use fp16. Unsloth picks this automatically; you never set a dtype yourself.

In [ ]:
import torch, subprocess
print(subprocess.check_output(["nvidia-smi"]).decode())
print("Device     :", torch.cuda.get_device_name(0))
props = torch.cuda.get_device_properties(0)
print(f"Total VRAM : {props.total_memory/1024**3:.2f} GB")
print("bf16 OK?   :", torch.cuda.is_bf16_supported())  # -> False on T4 (Turing); we use fp16

### Step 3 · Load Qwen2.5-1.5B (with three memory tricks)

Three ideas let a 1.5B model **train *and* generate** inside 16 GB:
- **4-bit (`load_in_4bit`)** — store the base weights in 4 bits instead of 16. ~4× smaller.
- **LoRA (next cell)** — freeze the big model; train tiny "adapter" matrices instead. We update <1% of the weights.
- **`fast_inference` (vLLM)** — a separate fast engine for *generating* the practice answers GRPO will score.

`gpu_memory_utilization = 0.6` hands ~60% of VRAM to vLLM and keeps the rest for training. If you ever hit an out-of-memory error, lower it to `0.5`; if generation feels slow and you don't OOM, try `0.7`.

*(Memory tight? Swap `model_name` to `"Qwen/Qwen2.5-0.5B-Instruct"` — same cells, more headroom.)*

In [ ]:
from unsloth import FastLanguageModel
import torch

max_seq_length = 1024   # official reasoning-notebook default; raise only if VRAM allows
lora_rank      = 32     # safe T4 value for 1.5B (bigger GPUs use 64). Must match r in the next cell.

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name             = "Qwen/Qwen2.5-1.5B-Instruct",
    max_seq_length         = max_seq_length,
    load_in_4bit           = True,    # 4-bit base weights -> fits T4 (16GB)
    fast_inference         = True,    # vLLM-backed generation (GRPO needs fast rollouts)
    max_lora_rank          = lora_rank,
    gpu_memory_utilization = 0.6,     # T4-safe: vLLM grabs ~60% VRAM, leaves room to train
)

Now attach the trainable **LoRA adapters** — these small matrices are the *only* thing GRPO will update. The 1.5B base stays frozen.

In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r = lora_rank,                      # = max_lora_rank above
    target_modules = [
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ],
    lora_alpha = lora_rank,
    use_gradient_checkpointing = "unsloth",  # Unsloth long-context checkpointing — keep on
    random_state = 3407,
)

### Step 4 · Sanity generation

Generate one answer through the vLLM path to prove everything is wired up. We pass `lora_request = None` because there's no trained adapter yet. Watch how the **base** model reasons right now — in Phase 2 we'll measure exactly how good (or bad) it is, and that becomes our **"before"** snapshot.

> Later (Phase 5) you'll generate *with* the trained brain by changing one argument: `lora_request = model.load_lora("grpo_saved_lora")`.

In [ ]:
from vllm import SamplingParams

messages = [
    {"role": "system", "content": "You are a helpful assistant. Reason step by step."},
    {"role": "user",   "content": "What is 17 * 24? Show your reasoning, then give the final answer."},
]
text = tokenizer.apply_chat_template(
    messages, tokenize = False, add_generation_prompt = True,
)

sampling_params = SamplingParams(temperature = 0.8, top_p = 0.95, max_tokens = 512)

output = model.fast_generate(
    text,
    sampling_params = sampling_params,
    lora_request    = None,   # base model; no trained LoRA exists yet (that comes in Phase 4)
)[0].outputs[0].text

print(output)

---

## ✅ Phase 0 done

You have a **T4 GPU**, **Unsloth + vLLM** installed, **Qwen2.5-1.5B** loaded in 4-bit with **LoRA adapters**, and a **working generation path** — the whole RL *engine*, ready to be pointed at a task and a reward.

## Phase 1 — the task: Countdown + the reasoning format

The project's logic lives in three small, **unit-tested** Python files (this is what makes it resume-grade — not just notebook cells):
- **`task.py`** — generate solvable Countdown problems, the prompt template, and a *safe* answer checker.
- **`reward.py`** — the GRPO reward functions (Phase 3/4).
- **`evaluate.py`** — base-vs-trained scoring (Phases 2 & 5).

### 📤 Upload the modules
Open Colab's **Files** panel (folder icon, far left) → **Upload to session storage** → add `task.py`, `reward.py`, `evaluate.py` from your `cogito/` folder. *(If Colab disconnects and you reconnect, re-upload them and re-run from here.)*

### The format that makes reasoning gradeable
Every answer must look like this:
```
<reasoning> ...try combinations and check them... </reasoning>
<answer> (9 * 50) + (3 * 7) </answer>
```
The reward (Phase 3) reads `<answer>` to check correctness and the tags to check format. **No fixed format → nothing to grade.**

In [ ]:
import task, reward, evaluate   # the files you just uploaded
import random

# Peek at a generated problem + the exact prompt the model will see:
p = task.generate_problem(rng=random.Random(0))
print("Problem:", p)
print("\n--- prompt the model receives ---")
for m in task.make_prompt(p["nums"], p["target"]):
    print(f"[{m['role']}]\n{m['content']}\n")

# The answer checker is pure + SAFE (parses with Python's ast, never eval()):
print("check (9*50)+(3*7) for [3,7,9,50] -> 471 :",
      task.is_correct("(9 * 50) + (3 * 7)", [3, 7, 9, 50], 471))

## Phase 2 — baseline: the model *before* training (the "before")

Build the datasets, then measure the base model on **100 held-out problems** (kept **disjoint** from training so accuracy isn't memorisation). Expect a **low** score — that's the bar GRPO must beat.

> ⏱️ This generates 100 completions and takes a few minutes on a T4. Short on session time? Lower the eval `n` (e.g. 50).

In [ ]:
# Training set + a DISJOINT held-out eval set (no overlap = no cheating).
train_dataset = task.build_dataset(n=1000, seed=0)
train_keys = {task.problem_key({"nums": r["nums"], "target": r["target"]}) for r in train_dataset}
eval_dataset = evaluate.make_eval_dataset(n=100, seed=2025, exclude=train_keys)
print(f"train: {len(train_dataset)} problems  |  held-out eval: {len(eval_dataset)} problems (disjoint)")

In [ ]:
# BASELINE: base model, no trained adapter. This is our "before" number.
base_eval = evaluate.run_eval(model, tokenizer, eval_dataset,
                              lora_request=None, max_new_tokens=704, temperature=0.8)
print(f"\nBASE accuracy: {base_eval['accuracy']*100:.1f}%  ({base_eval['n_correct']}/{base_eval['n']})")

# Peek at two attempts to see the *style* of the untrained model:
for r in base_eval["records"][:2]:
    print("\n" + "="*60)
    print("nums", r["nums"], " target", r["target"], " -> correct:", r["correct"])
    print(r["completion"][:600])

evaluate.save_traces(base_eval["records"], "eval_base_traces.json")

## Phase 3 — the reward function (the heart) + what GRPO actually does

**GRPO in one picture:** for each problem the model writes `num_generations` attempts (a **group**). Each is scored. GRPO compares every attempt to the **group average** and shifts the model toward the above-average ones — no "correct reasoning" labels needed, just a number that's higher when the answer is right and well-formed.

`reward.py` sums four signals:

| reward | when it fires | why |
|---|---|---|
| **correctness** +2.0 | `<answer>` legally hits the target | the real objective |
| **partial** +0.5 | a legal expression using the given numbers (even if wrong) | shaping |
| **strict format** +0.5 | exact `<reasoning>/<answer>` structure | clean, extractable output |
| **soft format** +0.25 | both tags present, in order | easiest first win |

**Why shaping?** Correctness alone is too sparse early (every attempt scores 0 → no gradient). The format/partial rewards build a *ladder*: gibberish → right shape → legal expression → correct answer.

In [ ]:
# See the four rewards fire on a GOOD vs a BAD answer.
# (completions arrive in TRL's conversational shape: a list of message dicts.)
good = [[{"role": "assistant", "content": "<reasoning>9*50=450 and 3*7=21, so 450+21=471</reasoning>\n<answer>(9 * 50) + (3 * 7)</answer>"}]]
bad  = [[{"role": "assistant", "content": "hmm the answer is maybe 471?"}]]
for name, c in [("GOOD", good), ("BAD ", bad)]:
    print(name,
          "| correctness", reward.correctness_reward_func(None, c, target=[471], nums=[[3, 7, 9, 50]]),
          "| partial", reward.partial_reward_func(None, c, target=[471], nums=[[3, 7, 9, 50]]),
          "| strict_fmt", reward.strict_format_reward_func(c),
          "| soft_fmt", reward.soft_format_reward_func(c))

## Phase 4 — GRPO training (watch the reward climb)

Log in to **Weights & Biases**, set the **T4-tuned** config, and train.

**What to watch in W&B**
- `reward` (and `reward/<func>/mean`) — noisy, often **flat for the first 100–200 steps**, then it should **rise and plateau**. Judge progress by **reward**, *not* `loss` (GRPO's loss isn't a quality signal).
- `frac_reward_zero_std` → if it heads to 1, every attempt in a group scores the same (no learning signal).
- `completions/mean_length` — often grows as the model learns to "think" longer.

⏱️ ~1–2 h for 250 steps on a free T4 — keep the tab active. **OOM?** lower `num_generations` (8→6→4), then `max_completion_length`.

In [ ]:
# Weights & Biases login via a Colab SECRET (no key in code).
# In Colab: click the key icon (left) -> add a secret named WANDB_API_KEY -> enable notebook access.
import os
from google.colab import userdata
import wandb

os.environ["WANDB_API_KEY"] = userdata.get("WANDB_API_KEY")
os.environ["WANDB_PROJECT"] = "cogito-countdown-grpo"
wandb.login()   # logging turns on via report_to="wandb" in the config below

In [ ]:
from trl import GRPOConfig, GRPOTrainer

training_args = GRPOConfig(
    use_vllm = True,                       # generate rollouts with the fast vLLM engine
    temperature = 1.0,                     # exploration during rollouts (DeepSeek-R1 style)
    learning_rate = 5e-6,                  # low LR -> stable GRPO
    adam_beta1 = 0.9, adam_beta2 = 0.99,
    weight_decay = 0.1, warmup_ratio = 0.1, lr_scheduler_type = "cosine",
    optim = "adamw_8bit",                  # 8-bit optimizer saves VRAM on the T4
    max_grad_norm = 0.1,
    per_device_train_batch_size = 1,
    gradient_accumulation_steps = 1,
    num_generations = 8,                   # GROUP SIZE (attempts per problem). Drop 8->6->4 if OOM.
    max_prompt_length = 320,               # room for our system + user prompt
    max_completion_length = 704,           # 320 + 704 = 1024 = max_seq_length
    max_steps = 250,                       # ~1-2 h on a free T4 (raise for stronger results)
    fp16 = True, bf16 = False,             # T4 has no bf16
    logging_steps = 1,                     # GRPO curves are noisy; log every step
    save_steps = 250,
    report_to = "wandb",                   # <- sends the reward curve to W&B
    output_dir = "outputs",
)

In [ ]:
trainer = GRPOTrainer(
    model = model,
    processing_class = tokenizer,          # current TRL uses processing_class (NOT tokenizer=)
    reward_funcs = reward.REWARD_FUNCS,    # our 4 reward functions (summed per attempt)
    args = training_args,
    train_dataset = train_dataset,
)
trainer.train()   # watch reward/<func_name>/mean climb in the W&B dashboard

## Phase 5 — evaluate the trained model (the "after")

Save the trained LoRA, re-run the **same 100 held-out problems** with it, and write the **before/after** report. The one-line change that uses the trained brain is `lora_request = model.load_lora("grpo_saved_lora")`.

In [ ]:
# Save the trained LoRA, then evaluate it on the SAME 100 held-out problems.
model.save_lora("grpo_saved_lora")
lora = model.load_lora("grpo_saved_lora")

trained_eval = evaluate.run_eval(model, tokenizer, eval_dataset,
                                 lora_request=lora, max_new_tokens=704, temperature=0.8)

print(evaluate.summarize(base_eval, trained_eval))          # Base X% -> Trained Y% (+Z pts)
evaluate.save_traces(trained_eval["records"], "eval_trained_traces.json")
evaluate.save_before_after(base_eval["records"], trained_eval["records"], "before_after.md")
print("wrote before_after.md  (your aha-moment artifact)")

## Phase 6 — push to Hugging Face Hub

Push the trained **LoRA adapter** (small — it re-attaches to the base model). Add a Colab secret **HF_TOKEN** (write scope) first.

In [ ]:
# Push the trained LoRA adapter (a few MB) to the Hugging Face Hub.
from google.colab import userdata
from huggingface_hub import login
login(token=userdata.get("HF_TOKEN"))

HF_USERNAME = "YOUR_HF_USERNAME"            # <-- edit me
repo = f"{HF_USERNAME}/cogito-countdown-grpo"
model.push_to_hub(repo)
tokenizer.push_to_hub(repo)
print("pushed LoRA to:", repo)

# Optional, heavier (~3 GB): a standalone merged 16-bit model
# model.push_to_hub_merged(f"{HF_USERNAME}/cogito-countdown-grpo-merged", tokenizer, save_method="merged_16bit")

## Phase 7 (optional) — a live demo

`app.py` (in the repo) is a small **Gradio** app: enter numbers + a target and watch the trained model reason. To deploy as a **Hugging Face Space**:
1. Create a new **Gradio** Space (a **GPU** Space is recommended).
2. Upload `app.py` and a `requirements.txt` with `gradio` and `unsloth`.
3. Edit `MODEL_ID` in `app.py` to your pushed repo (`YOUR_HF_USERNAME/cogito-countdown-grpo`).

---

## 🎉 Done — tell the story
You trained a 1.5B model to **reason** with GRPO and measured it: **base → trained accuracy** on 100 held-out Countdown problems, with `before_after.md` showing the *same* problems solved before vs after. Drop that number + one before/after example into `README.md` — that's your headline.